# Mini FSD training on Colab

Thin by design (ADR 0007): this notebook mounts Drive, installs the repo at a
pinned commit, unpacks the data and calls a `minifsd.train` module. All the
logic is in the repo, tested locally.

Before the first run, upload the packs from `data/packs/` (made by
`python -m minifsd.train.pack`) to `My Drive/mini-fsd/packs/`.

Runtime > Change runtime type > T4 GPU. If the session drops, run all cells
again: training resumes from the last checkpoint on Drive.

In [1]:
# Settings for this run. COMMIT pins the code, so a result can be traced
# back to it; use the commit you pushed last (git rev-parse --short HEAD).
COMMIT = "main"
PACKS = ["v9.zip", "coco-people.zip"]
DRIVE = "/content/drive/MyDrive/mini-fsd"
RUN = "e2a-segformer-b0"  # checkpoints and logs go to DRIVE/runs/RUN

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# The repo at the pinned commit. Extra packages, pinned, are added here per
# experiment and recorded in ADR 0007.
# --force-reinstall: a new COMMIT must replace the old install even though
# the version number is the same. --no-deps: keep Colab's numpy and scipy.
!pip install -q --force-reinstall --no-deps "git+https://github.com/SegunAdewola/mini-fsd@{COMMIT}"

In [ ]:
import pathlib
import shutil

from minifsd.train import pack

data = pathlib.Path("/content/data")
data.mkdir(exist_ok=True)
for name in PACKS:
    local = data / name
    if not local.exists():
        shutil.copy(f"{DRIVE}/packs/{name}", local)
    folder = data / name.removesuffix(".zip")
    if not folder.exists():
        print("unpacked", pack.unpack(local, data))

## E2a: SegFormer-B0 floor segmentation

Smoke run first: 64 frames, one epoch, on the session disk. It checks the
whole loop on the GPU in a minute or two before the real run.

In [ ]:
!python -m minifsd.train.segformer --train /content/data/v9/train --val /content/data/v9/val --run-dir /content/smoke --epochs 1 --limit 64

The real run: all 9,000 v9 train frames, 20 epochs, checkpoints on Drive.
If the session drops, run every cell again from the top: it resumes after
the last finished epoch.

In [ ]:
!python -m minifsd.train.segformer --train /content/data/v9/train --val /content/data/v9/val --run-dir {DRIVE}/runs/{RUN} --epochs 20

In [ ]:
# One line per finished epoch; best/ holds the highest val IoU so far.
!cat {DRIVE}/runs/{RUN}/log.jsonl